In [ ]:
import sys
!{sys.executable} -m pip install graphviz

In [ ]:
import sys
!{sys.executable} -m pip install ipywidgets

In [ ]:
!brew install graphviz

In [ ]:
import math

In [ ]:
import random

In [ ]:
import graphviz
print(graphviz.__version__)

In [51]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)   # ← fixed: was self._pre
        self._op = _op
        self.grad = 0.0
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, _children=(self, other), _op='+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, _children=(self, other), _op='*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, n):
        assert isinstance(n, (int, float)), "exponent must be a number"
        out = Value(self.data ** n, _children=(self,), _op=f'**{n}')
        def _backward():
            self.grad += (n * (self.data ** (n - 1))) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), _children=(self,), _op='exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, _children=(self,), _op='tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), _children=(self,), _op='relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out
    def sigmoid(self):
        s = 1 / (1 + math.exp(-self.data))
        out = Value(s, _children=(self,), _op='sigmoid')
        def _backward():
            self.grad += s * (1 - s) * out.grad  # s(1-s)
        out._backward = _backward
        return out

    def leaky_relu(self, alpha=0.01):
        r = self.data if self.data > 0 else alpha * self.data
        out = Value(r, _children=(self,), _op='leaky_relu')
        def _backward():
            self.grad += (1.0 if self.data > 0 else alpha) * out.grad
        out._backward = _backward
        return out
    
    def swish(self):
        s = 1 / (1 + math.exp(-self.data))
        out = Value(self.data * s, _children=(self,), _op='swish')
        def _backward():
            s2 = 1 / (1 + math.exp(-self.data))
            self.grad += (s2 + self.data * s2 * (1 - s2)) * out.grad
        out._backward = _backward
        return out
    def log(self):
        out = Value(math.log(self.data), _children=(self,), _op='log')
        def _backward():
            self.grad += (1 / self.data) * out.grad
        out._backward = _backward
        return out
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __sub__(self, other):  return self + (other * -1)
    def __neg__(self):         return self * (-1)
    def __truediv__(self, other): return self * (other ** -1)

    def backward(self):
        topo = []
        visited = set()
        def dfs(node):
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    dfs(child)
                topo.append(node)
        dfs(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = graphviz.Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    
    nodes, edges = trace(root)
    
    for n in nodes:
        # draw a rectangle for every Value node
        dot.node(name=str(id(n)), 
                 label=f"data {n.data:.4f} | grad {n.grad:.4f}", 
                 shape='record')
        
        if n._op:
            # draw a circle for the operation
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [ ]:
class Loss:
    
    @staticmethod
    def mse(ypred, ytrue):
        """Mean Squared Error — regression problems"""
        return sum((yp - yt)**2 for yp, yt in zip(ypred, ytrue)) * (1 / len(ypred))

    @staticmethod
    def mae(ypred, ytrue):
        """Mean Absolute Error — robust to outliers"""
        return sum((yp - yt) if (yp - yt).data > 0 else (yt - yp) 
                   for yp, yt in zip(ypred, ytrue)) * (1 / len(ypred))

    @staticmethod
    def binary_cross_entropy(ypred, ytrue):
        """Binary Cross Entropy — binary classification (targets must be 0 or 1)"""
        eps = 1e-7  # prevent log(0)
        loss = Value(0.0)
        for yp, yt in zip(ypred, ytrue):
            # clamp prediction between eps and 1-eps
            yp_clamped = Value(min(max(yp.data, eps), 1 - eps))
            loss = loss + (
                Value(-yt.data) * yp_clamped.log() +
                Value(-(1 - yt.data)) * (Value(1.0) - yp_clamped).log()
            )
        return loss * (1 / len(ypred))

    @staticmethod
    def hinge(ypred, ytrue):
        """Hinge Loss — SVM-style (targets must be -1 or +1)"""
        losses = []
        for yp, yt in zip(ypred, ytrue):
            margin = Value(1.0) - Value(yt.data) * yp
            losses.append(Value(max(0, margin.data)))
        return sum(losses) * (1 / len(losses))

In [ ]:
class DataLoader:
    def __init__(self, xs, ys, batch_size=2, shuffle=True):
        self.xs = xs
        self.ys = ys
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        indices = list(range(len(self.xs)))
        if self.shuffle:
            random.shuffle(indices)
        
        # yield batches one at a time
        for start in range(0, len(indices), self.batch_size):
            batch_idx = indices[start : start + self.batch_size]
            yield (
                [self.xs[i] for i in batch_idx],
                [self.ys[i] for i in batch_idx]
            )

    def __len__(self):
        return len(self.xs) // self.batch_size

In [ ]:
class Neuron:
    def __init__(self,n):
        #n=number of inputs
        self.w=[Value(random.uniform(-1,1)) for _ in range(n)]
        self.b=Value(random.uniform(-1,1))
    def __call__(self,x):
        act=sum((wi*xi for wi,xi in zip(self.w,x)), self.b)
        return act.tanh()
    def parameters(self):
        return self.w+[self.b]
    def __repr__(self):
        return f"number of inputs={len(self.w)}"
        

In [ ]:
class Layer:
    def __init__(self, ni, no):
        #ni,no=no.of inputs, no.of outputs
        self.neurons=[Neuron(ni) for _ in range(no)]
    def __call__(self,x):
        outs=[n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]
        
    def __repr__():
        return f"Layer(neurons={len(self.neurons)})"

In [ ]:
class MLP:
    def __init__(self,ni,ls):
        #ls=layer sizes
        sizes=[ni]+ls
        self.layers=[Layer(sizes[i],sizes[i+1]) for i in range(len(ls))]
    def __call__(self,x):
        for layer in self.layers:
            x=layer(x)
        return x
    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]
    def __repr__():
        return f"MLP(layers={len(self.layers)})"

In [53]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── build & train a fresh model ──────────────────────────────────────────────
xs_train = [
    [Value(2.0),  Value(3.0),  Value(-1.0)],
    [Value(3.0),  Value(-1.0), Value(0.5)],
    [Value(0.5),  Value(1.0),  Value(1.0)],
    [Value(1.0),  Value(1.0),  Value(-1.0)],
]
ys_train = [Value(1.0), Value(-1.0), Value(-1.0), Value(1.0)]

demo_model = MLP(3, [4, 4, 1])

def train_model(epochs, lr):
    for _ in range(epochs):
        ypred = [demo_model(x) for x in xs_train]
        loss  = Loss.mse(ypred, ys_train)
        for p in demo_model.parameters():
            p.grad = 0.0
        loss.backward()
        for p in demo_model.parameters():
            p.data -= lr * p.grad
    return loss.data

# ── widgets ───────────────────────────────────────────────────────────────────
title = widgets.HTML("<h2>🧠 Micrograd Interactive Demo</h2>")

epoch_slider = widgets.IntSlider(
    value=10, min=1, max=100, step=1,
    description='Epochs:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

lr_slider = widgets.FloatSlider(
    value=0.05, min=0.001, max=0.2, step=0.001,
    description='Learning Rate:',
    readout_format='.3f',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

x1 = widgets.FloatText(value=1.0,  description='Input x1:')
x2 = widgets.FloatText(value=0.5,  description='Input x2:')
x3 = widgets.FloatText(value=-1.0, description='Input x3:')

train_btn   = widgets.Button(description='🚀 Train Model',   button_style='success')
predict_btn = widgets.Button(description='🔮 Predict',       button_style='info')
reset_btn   = widgets.Button(description='🔄 Reset Model',   button_style='warning')

output = widgets.Output()

# ── button logic ─────────────────────────────────────────────────────────────
def on_train(b):
    with output:
        clear_output()
        loss = train_model(epoch_slider.value, lr_slider.value)
        print(f"✅ Training complete!")
        print(f"   Epochs:        {epoch_slider.value}")
        print(f"   Learning rate: {lr_slider.value:.3f}")
        print(f"   Final loss:    {loss:.6f}")

def on_predict(b):
    with output:
        clear_output()
        x = [Value(x1.value), Value(x2.value), Value(x3.value)]
        pred = demo_model(x)
        label = "✅ Positive (≈ +1)" if pred.data > 0 else "❌ Negative (≈ -1)"
        print(f"🔮 Prediction for [{x1.value}, {x2.value}, {x3.value}]")
        print(f"   Raw output: {pred.data:.4f}")
        print(f"   Label:      {label}")

def on_reset(b):
    global demo_model
    demo_model = MLP(3, [4, 4, 1])
    with output:
        clear_output()
        print("🔄 Model reset — weights re-randomized!")

train_btn.on_click(on_train)
predict_btn.on_click(on_predict)
reset_btn.on_click(on_reset)

# ── layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    title,
    widgets.HTML("<b>Training Settings</b>"),
    epoch_slider,
    lr_slider,
    widgets.HBox([train_btn, reset_btn]),
    widgets.HTML("<hr><b>Run a Prediction</b>"),
    widgets.HBox([x1, x2, x3]),
    predict_btn,
    widgets.HTML("<hr><b>Output</b>"),
    output
]))